In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"})
def merge_columns(row):
    return {"combined_text": f"{row['prompt']} {row['A']}"}

updated_dataset = dataset.map(merge_columns)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
row51_combined_text = updated_dataset["train"][51]["combined_text"]
len(row51_combined_text)

614

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [5]:
print(tokenizer.sep_token_id)

102


In [6]:
encoded_inputs = tokenizer(
    list(updated_dataset["train"]["prompt"]),
    max_length=128,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)
print(encoded_inputs["input_ids"].shape)

torch.Size([2000, 128])


In [7]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
inputs = tokenizer(dataset["train"]["prompt"][0],return_tensors="pt")
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state
print(last_hidden_states.shape)

torch.Size([1, 31, 768])


In [9]:
cls_vector = last_hidden_states[0 , 0 , :]
print(cls_vector[0:5])
sum_5 = cls_vector[0:5].sum()
print(sum_5)

tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480], grad_fn=<SliceBackward0>)
tensor(-1.2001, grad_fn=<SumBackward0>)


In [10]:
tokenizer2 = AutoTokenizer.from_pretrained("bert-base-uncased")
model2 = AutoModel.from_pretrained("bert-base-uncased",output_attentions=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
string1 = "Light-ion fusion is a technique."
encoding = tokenizer2(string1, return_tensors="pt")
outputs = model2(**encoding)
attention_matrix = outputs.attentions[-1][0,0]
attention_matrix

tensor([[0.1148, 0.0409, 0.0698, 0.0579, 0.1025, 0.0245, 0.0211, 0.1091, 0.3029,
         0.1566],
        [0.0054, 0.0203, 0.0261, 0.0314, 0.0172, 0.0015, 0.0016, 0.0062, 0.7036,
         0.1868],
        [0.0030, 0.0169, 0.0210, 0.0260, 0.0210, 0.0026, 0.0026, 0.0060, 0.7046,
         0.1963],
        [0.0038, 0.0074, 0.0094, 0.0102, 0.0080, 0.0014, 0.0011, 0.0021, 0.7684,
         0.1882],
        [0.0076, 0.0101, 0.0214, 0.0278, 0.0436, 0.0030, 0.0032, 0.0079, 0.6978,
         0.1774],
        [0.0208, 0.0280, 0.0470, 0.0395, 0.0672, 0.0144, 0.0119, 0.0348, 0.5359,
         0.2005],
        [0.0118, 0.0293, 0.0409, 0.0384, 0.0636, 0.0111, 0.0089, 0.0310, 0.5714,
         0.1937],
        [0.0138, 0.0233, 0.0498, 0.0391, 0.1433, 0.0079, 0.0076, 0.0322, 0.5015,
         0.1815],
        [0.0042, 0.0038, 0.0023, 0.0025, 0.0041, 0.0028, 0.0018, 0.0027, 0.7664,
         0.2094],
        [0.0033, 0.0034, 0.0021, 0.0027, 0.0040, 0.0018, 0.0012, 0.0022, 0.7872,
         0.1920]], grad_fn=<